# 03 — Event-level extraction

Three parameter-level pulls answering questions notebook 02 raised but could not settle: what monetization looks like across a player's life, how scores distribute by mode, and what the seven DAU spike days were made of.

**Source:** the same public export as notebook 01. Parameters live in a repeated `event_params` field, reached with `CROSS JOIN UNNEST`.
**Produces:** `monetization_events.csv` (1,939 rows), `post_score_distribution.csv` (312), `spike_forensics.csv` (853).
**Consumed by:** notebook 04 (spike forensics) and notebook 05 (score, mode, monetization).

Design-first: every pull was specced and argued before any of them ran, so each query is minimal and answers exactly one question. `day_0.csv` loads below as the single source of truth for cohort membership — it is never recomputed in SQL.

In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="flooit-analytics-project")

In [3]:
import pandas as pd

In [4]:
day_0 = pd.read_csv('day_0.csv', parse_dates = ['day_0'])

print(day_0.shape)
print(day_0.dtypes)
day_0.head()

(4319, 2)
user_pseudo_id            object
day_0             datetime64[ns]
dtype: object


,user_pseudo_id,day_0
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15
2,6B41795D5E5B7339E007330941C9E201,2018-06-15
3,0FB42D5FFB79A9DE147873753BF7A664,2018-07-15
4,61C77C8D9E7F92524289DA7E9D4786BF,2018-07-15


## Pull 2 — Monetization lifecycle

One wide row per monetization event: 1,912 `ad_reward` and 27 `in_app_purchase`. The question is *"when in a player's life money appears?"*, so the extraction stays at event grain and the cohort clock attaches in pandas rather than in SQL.

### How the query works

GA4 stores each parameter value in one of four typed slots — string, int, float, double — and NULLs the rest. `COALESCE` across all four recovers the value whatever its type; casting to STRING gives the pipeline one column type to carry. Missing a slot means silently dropping every event logged on the platform that used it.

The `pivoted` stage turns one row per event-parameter into one row per event, one column per parameter. Parameters are not guaranteed present on every event, so `MAX(IF(key = 'x', value, NULL))` returns the value where the key exists and NULL where it doesn't — never an error, never a dropped row.

Types and units are stamped in the final `SELECT`, not mid-pipeline. Play and GA4 store money in micros, so `price` divides by 1e6 there and nowhere else. `value` is polysemous — reward steps on `ad_reward` rows, price in micros on purchase rows — so it is cast for convenience but never summed across event types.

In [5]:
query = """WITH params AS (SELECT user_pseudo_id,
                       event_date,
                       event_timestamp,
                       event_name,
                       param.key AS parameter_name,
                       COALESCE(param.value.string_value,
                                CAST(param.value.int_value    AS STRING),
                                CAST(param.value.float_value  AS STRING),
                                CAST(param.value.double_value AS STRING)) AS param_value
                FROM `firebase-public-project.analytics_153293282.events_*` CROSS JOIN UNNEST(event_params) AS param
                WHERE event_name IN ('in_app_purchase', 'ad_reward')),

     pivoted AS (SELECT user_pseudo_id, event_date, event_timestamp, event_name,
                     MAX(IF(parameter_name = 'product_id', param_value, NULL)) AS product_id,
                     MAX(IF(parameter_name = 'price', param_value, NULL)) AS price,
                     MAX(IF(parameter_name = 'currency', param_value, NULL)) AS currency,
                     MAX(IF(parameter_name = 'validated', param_value, NULL)) AS validated,
                     MAX(IF(parameter_name = 'type', param_value, NULL)) AS type,
                     MAX(IF(parameter_name = 'value', param_value, NULL)) AS value
                 FROM params
                 GROUP BY user_pseudo_id, event_date, event_timestamp, event_name)

SELECT user_pseudo_id, event_date, event_timestamp, event_name, product_id, SAFE_CAST(price AS FLOAT64)/1000000 AS price, currency, validated, type, SAFE_CAST(value AS FLOAT64) AS value
FROM pivoted
ORDER BY user_pseudo_id, event_timestamp"""

mon = client.query(query).to_dataframe()
print(mon.shape)

(1939, 10)


In [6]:
mon.to_csv("monetization_events.csv", index=False)

### Pull-time validation

The next two cells are extraction-time checks, not analysis. They confirm the row split and parameter coverage, profile the 34 ad events with a missing `type`, then attach `day_0` to size how much of the monetization data belongs to the cohort.

That cohort figure — 26.3% of events but 17 of the 27 buyers — is followed up properly in notebook 05. It sits here because it was the check that confirmed the pull had landed correctly.

In [7]:
mon = pd.read_csv('monetization_events.csv', parse_dates = ['event_date'])

print(mon.shape)
print(mon.dtypes)
display(mon.head(15))

display(mon.notna().sum())

display(mon[(mon.event_name == 'ad_reward') & (mon.type.isna())]['value'].describe())

(1939, 10)
user_pseudo_id             object
event_date         datetime64[ns]
event_timestamp             int64
event_name                 object
product_id                 object
price                     float64
currency                   object
validated                 float64
type                       object
value                     float64
dtype: object


,user_pseudo_id,event_date,event_timestamp,event_name,product_id,price,currency,validated,type,value
0,00AE0CA4117376AE083FD6AEA744CE88,2018-09-02,1535943983925000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
1,00AE0CA4117376AE083FD6AEA744CE88,2018-09-02,1535944080985000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
2,014EE05605C94D797CA52C355638C967,2018-08-13,1534198373295000,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
3,014EE05605C94D797CA52C355638C967,2018-08-13,1534198445681031,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
4,014EE05605C94D797CA52C355638C967,2018-08-13,1534198451217008,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
5,025FF377B4148748AF5743A54761EA15,2018-06-26,1530010973066014,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
6,025FF377B4148748AF5743A54761EA15,2018-06-26,1530011082853096,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
7,025FF377B4148748AF5743A54761EA15,2018-06-26,1530011118403136,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
8,025FF377B4148748AF5743A54761EA15,2018-06-29,1530267199262014,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
9,026E9E99CF63F4AD31203F6B2E01949D,2018-07-12,1531454060475001,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0


,0
user_pseudo_id,1939
event_date,1939
event_timestamp,1939
event_name,1939
product_id,27
price,27
currency,27
validated,16
type,1878
value,1939


,value
count,34.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [8]:
cohort_share = pd.merge(day_0, mon, on='user_pseudo_id', how='inner')
display(cohort_share)

cohort_share_pct = round(100.0 * cohort_share.shape[0] / mon.shape[0], 2)
print(cohort_share_pct)

print(cohort_share.groupby('event_name').size())

,user_pseudo_id,day_0,event_date,event_timestamp,event_name,product_id,price,currency,validated,type,value
0,23D89EE594C105BFA999295B38C80B2B,2018-06-17,2018-06-25,1529969258260015,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
1,23D89EE594C105BFA999295B38C80B2B,2018-06-17,2018-06-25,1529969378807111,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
2,CEA79F13743A84BC8ABA238AB7D2851A,2018-07-29,2018-07-29,1532887607312172,ad_reward,NaN,NaN,NaN,NaN,NaN,1.0
3,C8B06F76EE9091107007C19B08C7834E,2018-07-07,2018-07-07,1531013957345055,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
4,C8B06F76EE9091107007C19B08C7834E,2018-07-07,2018-07-07,1531014293124152,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
...,...,...,...,...,...,...,...,...,...,...,...
505,22430BD1EDFC8E2E980313F10148A53E,2018-06-29,2018-07-04,1530725088446000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
506,A16DE718D8908FFC01DD297FEA600583,2018-09-17,2018-09-17,1537242133741001,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
507,1C0AF5475798F990F25A06C5A162E8DF,2018-07-30,2018-07-30,1532963380668009,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
508,1C0AF5475798F990F25A06C5A162E8DF,2018-07-30,2018-07-30,1532984265537000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0


26.3
event_name
ad_reward          493
in_app_purchase     17
dtype: int64


## Pull 3 — Score distribution

Collapses 242,039 `post_score` events into a 312-row distribution over (level, score, mode). Uses the whole player base rather than the cohort: this is a question about game mechanics, and mechanics have no install clock.

### How the query works

Three stages: unnest parameters, pivot to one row per event, collapse to counts.

The `CASE` on the final `SELECT` encodes a deduction rather than a guess. `post_score` carries 31 distinct levels where `level_start` carries only 30. Every level that gets played posts a score, so `level_start`'s 30 values must all appear among `post_score`'s 31 — which leaves exactly one level that posts scores but is never started, and it is level 0. Two independent checks agree: `level_start_quickplay` carries no `level` parameter at all, only a board size, so quickplay's posts need a bucket that is not a level; and level 0 holds 206,375 of 242,039 posts (85.3%), against quickplay's 87.6% share of level starts.

The event key `(user_pseudo_id, event_timestamp)` is deliberately absent from both the final `SELECT` and its `GROUP BY`. Keeping it would return one row per event instead of a distribution — the fix is removing columns from the SELECT, not adding them to the GROUP BY.

In [9]:
query = """WITH params AS (SELECT user_pseudo_id,
                       event_timestamp,
                       param.key AS parameter_name,
                       COALESCE(param.value.string_value,
                                CAST(param.value.int_value    AS STRING),
                                CAST(param.value.float_value  AS STRING),
                                CAST(param.value.double_value AS STRING)) AS param_value
                FROM `firebase-public-project.analytics_153293282.events_*` CROSS JOIN UNNEST(event_params) AS param
                WHERE event_name = 'post_score'),

     pivoted AS (SELECT user_pseudo_id, event_timestamp,
                     MAX(IF(parameter_name = 'level', param_value, NULL)) AS level,
                     MAX(IF(parameter_name = 'score', param_value, NULL)) AS score
                 FROM params
                 GROUP BY user_pseudo_id, event_timestamp)

SELECT level, score, CASE WHEN SAFE_CAST(level AS INT64) = 0 THEN 'quickplay' ELSE 'level_mode' END AS mode, COUNT(*) AS n_posts
FROM pivoted
GROUP BY level, score, mode
ORDER BY SAFE_CAST(level AS INT64), SAFE_CAST(score AS INT64)"""

ps = client.query(query).to_dataframe()
print(ps.shape)

(312, 4)


In [10]:
ps.to_csv("post_score_distribution.csv", index=False)

In [11]:
ps = pd.read_csv('post_score_distribution.csv')

print(ps.shape)
print(ps.dtypes)
ps.head()

(312, 4)
level       int64
score       int64
mode       object
n_posts     int64
dtype: object


,level,score,mode,n_posts
0,0,0,quickplay,56174
1,0,1,quickplay,52321
2,0,2,quickplay,43179
3,0,3,quickplay,29123
4,0,4,quickplay,15832


In [12]:
ps['n_posts'].sum()

np.int64(242039)

## Pull 4 — Spike forensics

Event-level composition of the seven DAU spike days against two references: the seven normal days interleaved between them, and the whole stable window from 2 Jul onward as a single anchor block.

Interleaving is the point — comparing spike days to days sitting *between* them controls for drift in the population mix as the window ages. Weekday matching was retired after notebook 02 showed the weekly rhythm was noise.

### How the query works

`window_group` is decided per user across the whole window, not per day. A user who ever logs `first_open` installed in-window; everyone else was already installed before 12 Jun. `LOGICAL_OR` makes that a single stable label, so the group being compared cannot drift between days — without it, a user could count as in-window on one date and pre-window on another and the comparison would be meaningless.

Two arms are unioned. The 14 spike and interleaved days stay per-date; everything from 2 Jul collapses into a single `stable_block` anchor row. Day labels attach in pandas rather than here, keeping the extraction thin and leaving analysis decisions to the analysis.

One consequence: `period` carries two semantic types — date strings and a block label. Columns like this never get a single type stamp, which is why it loads with `dtype={'period': str}` in notebook 04 rather than `parse_dates`.

In [13]:
query = """WITH user_window AS (SELECT user_pseudo_id, IF(LOGICAL_OR(event_name = 'first_open'), 'in_window', 'pre_window') AS window_group
                     FROM `firebase-public-project.analytics_153293282.events_*`
                     GROUP BY user_pseudo_id),

     labeled AS (SELECT e.user_pseudo_id, event_date, event_name, window_group
                 FROM `firebase-public-project.analytics_153293282.events_*` AS e JOIN user_window AS uw
                 ON e.user_pseudo_id = uw.user_pseudo_id),

     arm_dates AS (SELECT event_date AS period, window_group, event_name, COUNT(*) AS n_events, COUNT(DISTINCT user_pseudo_id) AS n_users
                   FROM labeled
                   WHERE event_date IN ('20180618','20180619','20180620','20180621','20180622','20180623','20180624','20180625','20180626','20180627','20180628','20180629','20180630','20180701')
                   GROUP BY period, window_group, event_name),

     arm_block AS (SELECT 'stable_block' AS period, window_group, event_name, COUNT(*) AS n_events, COUNT(DISTINCT user_pseudo_id) AS n_users
                   FROM labeled
                   WHERE event_date >= '20180702'
                   GROUP BY window_group, event_name)

SELECT * FROM arm_dates
UNION ALL
SELECT * FROM arm_block"""

sf = client.query(query).to_dataframe()
print(sf.shape)
print(sf.groupby(['period','window_group','event_name']).size().max())

(853, 5)
1


In [14]:
sf.to_csv("spike_forensics.csv", index=False)

In [15]:
sf = pd.read_csv('spike_forensics.csv')

print(sf.shape)
print(sf.dtypes)
sf.head()

(853, 5)
period          object
window_group    object
event_name      object
n_events         int64
n_users          int64
dtype: object


,period,window_group,event_name,n_events,n_users
0,20180629,in_window,screen_view,2675,60
1,20180629,in_window,post_score,203,41
2,20180629,pre_window,level_end_quickplay,2900,306
3,20180629,pre_window,level_complete_quickplay,1916,274
4,20180629,pre_window,screen_view,18186,418


## What this notebook establishes

Three CSVs, each reconciled against a known total before saving: monetization at 1,939 rows (1,912 + 27), the score distribution summing to exactly 242,039 posts with 206,375 in the level-0 bucket, and spike forensics at 853 rows verified duplicate-free on its grain.

**Not concluded here.** This notebook extracts and validates; it does not interpret. The spike verdict belongs to notebook 04, the score and monetization findings to notebook 05. One fact carries forward: `value` means different things on different rows — reward steps on ad events, price in micros on purchases — so it is never summed across event types.